In [2]:
import pandas as pd
from datetime import datetime
import xlsxwriter



In [8]:
import pandas as pd
from datetime import datetime
import xlsxwriter
from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils.dataframe import dataframe_to_rows

#===============================================
# Step 1. Load Excel file into pandas
#===============================================
file_path = r"C:\Users\awidjaja\Documents\RFL- POC\Input_Files\Sample 1_Transaction Register Analysis - 2025-10-21T094422.255.csv"
df = pd.read_csv(file_path)

#========================================================
# Step 2. Select and rename columns (based on VBA intention)
#========================================================
# Choose main useful fields — adjust to match business logic
df = df[[
    "Transaction Number",
    "Transaction Date",
    "Accounting Date",
    "Bill-to Customer Name",
    "Bill-to Customer Account Number",
    "Coverage Month",
    "Line Amount",
    "Transaction Type Name",
    "Comments"
]]

# Rename to user-friendly column names
df.rename(columns={
    "Transaction Number": "Invoice #",
    "Transaction Date": "Invoice Date",
    "Accounting Date": "Acct Date",
    "Bill-to Customer Name": "Customer Name",
    "Bill-to Customer Account Number": "Customer Account #",
    "Coverage Month": "Coverage Month",
    "Line Amount": "Amount",
    "Transaction Type Name": "Type",
    "Comments": "Comments"
}, inplace=True)

#========================================================
# Step 3. Sort by Amount ascending (like VBA sort on col G)
#========================================================
df.sort_values(by="Amount", ascending=True, inplace=True)

#========================================================
# Step 4. Export to Excel and apply formatting
#========================================================
todaydate = datetime.now().strftime("%m-%d-%Y_%H-%M-%S")
output_path = "../test/Team_Register_Output"+todaydate+".xlsx"
with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    df.to_excel(writer, index=False, sheet_name="Sheet1")


#===============================================
# Step 5. Apply Excel formatting using openpyxl
#===============================================
wb = load_workbook(output_path)
ws = wb["Sheet1"]

# Header formatting (light blue fill)
header_fill = PatternFill(start_color="BDD7EE", end_color="BDD7EE", fill_type="solid")
header_font = Font(bold=True)
thin = Side(border_style="thin", color="000000")
border = Border(top=thin, bottom=thin)

for cell in ws[1]:
    cell.fill = header_fill
    cell.font = header_font
    cell.border = border

# Autofit columns (approximate)
for col in ws.columns:
    max_length = 0
    col_letter = col[0].column_letter
    for cell in col:
        try:
            cell_length = len(str(cell.value))
            if cell_length > max_length:
                max_length = cell_length
        except:
            pass
    ws.column_dimensions[col_letter].width = max_length + 2

#===============================================
# Step 6. Add “Pmts”, “Applied”, “Balance” block (M2:N4)
#===============================================
ws["M2"] = "Pmts"
ws["M3"] = "Applied"
ws["M4"] = "Balance"

ws["N2"] = 0
ws["N3"] = 0
ws["N4"] = "=N2-N3"

for r in ws["M2:N4"]:
    for cell in r:
        cell.fill = header_fill
        cell.font = header_font
        cell.border = border

#===============================================
# Step 7. Apply autofilter
#===============================================
ws.auto_filter.ref = ws.dimensions

wb.save(output_path)
print(f"✅ Conversion complete. Saved to {output_path}")


✅ Conversion complete. Saved to ../test/Team_Register_Output11-03-2025_05-26-42.xlsx


In [6]:
file_path = r"C:\Users\awidjaja\Documents\RFL- POC\Input_Files\Sample 1_Transaction Register Analysis - 2025-10-21T094422.255.csv"
df22 = pd.read_csv(file_path)

In [2]:
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Border, Side
from datetime import datetime

#======================================================
# Step 1. Read the CSV or Excel input
#======================================================
file_path = r"C:\Users\awidjaja\Documents\RFL- POC\Input_Files\Sample 1_Transaction Register Analysis - 2025-10-21T094422.255.csv"
df = pd.read_csv(file_path)

#======================================================
# Step 2. Select & rename columns (mapping VBA intent)
#======================================================
# Adjust mapping to match your real headers
df = df[[
    "Transaction Line Description",
    "Transaction Date",
    "Transaction Number",
    "Bill-to Customer Name",
    "Bill-to Customer Account Number",
    "Coverage Month",
    "Line Amount",
    "Previous Transaction Number"
]]

df.columns = [
    "Description", "Invoice Date", "Invoice #", "Customer Name",
    "Customer Account #", "Coverage Month", "Amount", "Prev Transaction #"
]

#======================================================
# Step 3. Remove zero Amount rows (VBA V2.1)
#======================================================
df = df[df["Amount"] != 0]

#======================================================
# Step 4. Sort by Amount (then by Prev Transaction #)
#======================================================
df.sort_values(by=["Amount", "Prev Transaction #"], ascending=[True, True], inplace=True)

#======================================================
# Step 5. Create Subtotals grouped by Prev Transaction #
#======================================================
subtotals = (
    df.groupby("Prev Transaction #", dropna=False)["Amount"]
      .sum()
      .reset_index()
      .rename(columns={"Amount": "Subtotal"})
)

# Insert subtotal rows at the bottom of each group
subtotal_rows = []
for _, row in subtotals.iterrows():
    subtotal_rows.append({
        "Description": f"Total for {row['Prev Transaction #']}",
        "Invoice Date": "",
        "Invoice #": "",
        "Customer Name": "",
        "Customer Account #": "",
        "Coverage Month": "",
        "Amount": row["Subtotal"],
        "Prev Transaction #": row["Prev Transaction #"]
    })
df_out = pd.concat([df, pd.DataFrame(subtotal_rows)], ignore_index=True)

#======================================================
# Step 6. Write to Excel
#======================================================
todaydate = datetime.now().strftime("%m-%d-%Y_%H-%M-%S")
output_path = "../test/Team_Register_Output"+todaydate+".xlsx"
with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    df.to_excel(writer, index=False, sheet_name="Sheet1")

#======================================================
# Step 7. Formatting in openpyxl
#======================================================
wb = load_workbook(output_path)
ws = wb.active

header_fill = PatternFill(start_color="BDD7EE", end_color="BDD7EE", fill_type="solid")
header_font = Font(bold=True)
thin = Side(border_style="thin", color="000000")
border = Border(top=thin, bottom=thin)

# Header formatting
for cell in ws[1]:
    cell.fill = header_fill
    cell.font = header_font
    cell.border = border

# Bold subtotal rows (those beginning with “Total”)
for row in ws.iter_rows(min_row=2):
    cell = row[0]
    if isinstance(cell.value, str) and cell.value.startswith("Total"):
        for c in row:
            c.font = Font(bold=True)
            c.border = border

# Auto column width
for col in ws.columns:
    max_length = 0
    col_letter = col[0].column_letter
    for cell in col:
        try:
            max_length = max(max_length, len(str(cell.value)))
        except:
            pass
    ws.column_dimensions[col_letter].width = max_length + 2

#======================================================
# Step 8. Append “Pmts / Applied / Balance” block
#======================================================
last_row = ws.max_row + 2
ws[f"F{last_row}"] = "Pmts"
ws[f"F{last_row+1}"] = "Applied"
ws[f"F{last_row+2}"] = "Balance"
ws[f"G{last_row}"] = 0
ws[f"G{last_row+1}"] = f"=G{last_row-3}"  # mimic VBA =R[-3]C
ws[f"G{last_row+2}"] = f"=G{last_row}-G{last_row+1}"

for r in ws[f"F{last_row}:G{last_row+2}"]:
    for c in r:
        c.fill = header_fill
        c.font = header_font
        c.border = border

ws.auto_filter.ref = ws.dimensions
wb.save(output_path)
print(f"✅ File saved: {output_path}")

✅ File saved: ../test/Team_Register_Output11-03-2025_11-29-14.xlsx
